# ❤️ 02 — Aquisição e Curadoria: Cardiotoxicidade hERG (TDC)
**Projeto:** MultiTarget BioPredict (Triagem Virtual Multiobjetivo $A_{2A}$ × hERG)  
**Fase:** 1 — Aquisição e Curadoria de Dados  
**Autor:** Raphael Pires

---

## ⚠️ Contexto Toxicológico e Regulatório

O gene *human Ether-à-go-go-Related Gene* (*KCNH2*) codifica a subunidade alfa formadora de poro do canal de potássio dependente de voltagem $K_v11.1$ (**hERG**), responsável pela condução da corrente de repolarização rápida miocárdica ($I_{Kr}$).

- **Mecanismo de Bloqueio:** Fármacos de pequenas moléculas que se ligam à cavidade hidrofóbica interna do canal inibem a saída de potássio, prolongando a duração do potencial de ação ventricular e o **intervalo QT** no eletrocardiograma.
- **Gravidade Clínica:** O prolongamento de QT induz a arritmia ventricular potencialmente fatal **Torsades de Pointes (TdP)** e fibrilação ventricular. Diversos fármacos de alta potência (ex.: Terfenadina, Astemizol, Cisaprida) foram retirados do mercado mundial por essa toxicidade cardíaca silenciosa.
- **Regulação:** As diretrizes internacionais ICH S7A e S7B exigem a avaliação mandante do risco de bloqueio hERG em estágios precoces da descoberta de fármacos.

---

## 🔬 Escopo do Pipeline
Este caderno realiza o processo completo de curadoria de dados para o **Modelo 2 (Classificação Binária Calibrada de Cardiotoxicidade $P_{\text{hERG}}$)**:
1. Ingestão direta do conjunto de dados `herg_karim` (~13.4k compostos) do **Therapeutics Data Commons (TDC)** via Harvard Dataverse API.
2. Padronização química com **RDKit**: remoção de contraíons/sais, neutralização de cargas e conversão para **SMILES canônico isomérico**.
3. Auditoria estrutural de duplicatas e **resolução estrita de conflitos de classe**: identificação e descarte de moléculas que apresentam anotações opostas (ativas e inativas simultaneamente em diferentes fontes).
4. Avaliação do balanceamento de classes e mapeamento para as faixas de semáforo do PRD.
5. Exportação do dataset curado final para `data/processed/herg_curated.csv`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Adiciona o diretório raiz do projeto ao path para carregar módulos do src/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.chem_utils import canonicalize_smiles
from src.data_utils import download_tdc_dataset, curate_herg_dataset, RAW_DIR, PROCESSED_DIR

print(f"Diretório raiz do projeto: {project_root}")
print(f"Diretório de dados brutos: {RAW_DIR}")
print(f"Diretório de dados processados: {PROCESSED_DIR}")

## 1. Aquisição do Dataset TDC (Harvard Dataverse)

Utilizamos a função `download_tdc_dataset('herg_karim')` (`src/data_utils.py`), que realiza o download direto da fonte primária do Therapeutics Data Commons hospedada no Harvard Dataverse (File ID: 6822246).  
Essa abordagem **elimina o PyTDC**, evitando dependências incompatíveis no Windows (como `tiledbsoma`) e conflitos de versões de bibliotecas científicas.

In [ ]:
df_tdc_raw = download_tdc_dataset("herg_karim", dest_dir=RAW_DIR, force=False)
print(f"Total de registros brutos TDC herg_karim: {len(df_tdc_raw)}")
print("\nColunas do dataset:")
print(list(df_tdc_raw.columns))

# Amostra das primeiras linhas
df_tdc_raw.head()

## 2. Padronização Química via RDKit

A higienização molecular é realizada com o mesmo padrão RDKit utilizado para o receptor $A_{2A}$:
- Remoção de sais e contraíons inorgânicos/orgânicos (`LargestFragmentChooser`).
- Neutralização de grupos ácidos e básicos ionizados (`Uncharger`).
- Geração de SMILES canônico isomérico padronizado.

In [ ]:
print("Executando padronização química via RDKit...")
df_tdc_raw['clean_smiles'] = df_tdc_raw['Drug'].astype(str).apply(canonicalize_smiles)

valid_mask = df_tdc_raw['clean_smiles'].notna() & df_tdc_raw['Y'].notna()
df_tdc_clean = df_tdc_raw.loc[valid_mask].copy()

print(f"Registros brutos: {len(df_tdc_raw)}")
print(f"Registros com SMILES e rótulos válidos após padronização: {len(df_tdc_clean)} ({len(df_tdc_clean)/len(df_tdc_raw):.1%})")

## 3. Auditoria de Duplicatas e Resolução de Conflitos de Classe

Datasets compilados a partir de múltiplos ensaios de toxicidade (Patch-clamp manual, electrofisiologia automatizada, ensaios de ligação de radioligante) frequentemente contêm moléculas idênticas:
1. **Moléculas concordantes:** Moléculas testadas em múltiplos estudos com o mesmo resultado (ex.: todas apontando $Y=1$ ou todas apontando $Y=0$). Essas medições redundantes são fundidas em uma única instância consensual.
2. **Conflitos de classe:** Moléculas onde um ensaio reportou bloqueador ($Y=1$) e outro ensaio reportou não-bloqueador ($Y=0$). Para evitar ruído e contaminação nos rótulos de aprendizado de máquina, **descartamos integralmente os compostos ambíguos** ($0 < \text{média}(Y) < 1$).

In [ ]:
# Aplica a função de curadoria consolidada do src/data_utils.py
df_herg_curated = curate_herg_dataset(df_tdc_raw)

print(f"Moléculas únicas curadas no dataset final: {len(df_herg_curated)}")
print(f"Compostos com múltiplas anotações consensuais: {len(df_herg_curated[df_herg_curated['num_records'] > 1])}")
df_herg_curated.head(10)

## 4. Balanceamento de Classes e Resumo Toxicológico

Avaliamos a distribuição das classes binárias:
- $Y=1$: **Bloqueador hERG** (Cardiotóxico)
- $Y=0$: **Não-bloqueador** (Seguro)

In [ ]:
counts = df_herg_curated['hERG_label'].value_counts()
props = df_herg_curated['hERG_label'].value_counts(normalize=True).mul(100).round(2)

balance_df = pd.DataFrame({
    'Classe': ['Bloqueador hERG (Cardiotóxico)', 'Não-Bloqueador (Seguro)'],
    'Rótulo (Y)': [1, 0],
    'Total de Moléculas': [counts[1], counts[0]],
    'Percentual (%)': [props[1], props[0]]
})

print("Tabela de Balanceamento de Classes (Dataset Curado):")
display(balance_df)

## 5. Exportação do Dataset Curado

Gravamos o dataset final higienizado em `data/processed/herg_curated.csv`, pronto para featurização e modelagem.

In [ ]:
out_herg = PROCESSED_DIR / "herg_curated.csv"
export_cols = ["Drug_ID", "canonical_smiles", "hERG_label", "num_records"]
df_herg_curated[export_cols].to_csv(out_herg, index=False)

print(f"Dataset curado exportado com sucesso: {out_herg}")
print(f"Dimensões finais: {df_herg_curated[export_cols].shape}")
print(f"Colunas: {export_cols}")